# **Code for Training Arbitrary Unitary Transformation for $n=16$ Modes**

In [27]:
# general imports
import numpy as np
import matplotlib.pyplot as plt

# ml imports
import torch
import torch.nn.functional as F
from tqdm import tqdm

# import module
from nlse import *

## Parameters

In [42]:
# basis size
num_modes = 16

# Simulation parameters
Lz = 1e-3          # Propagation length
Nz = 100          # Number of z steps
Lt = 14         # Time window
Nt = 2048        # Number of time points
N_modes = 500      # Total HG basis size
dz = Lz / Nz
dt = Lt / Nt

# Medium parameters
beta2_j = -10.0
beta2_k = -10.0
gamma_j = 1.0
gamma_k = 1.0

# Pulse parameters
tau = 0.1
amplitude_downscale = 0.1

# penalty parameters
mask_percent = 0.8
m = 8              # Number of penalty iterations

# choose transformation
t_name = "permutation"

In [43]:
# Create time grid
t = torch.linspace(-Lt/2, Lt/2, Nt, dtype=torch.float32, device=device)

# Define HG basis
hg_basis = get_hg_basis(N_modes, t, tau)

Precomputed 500 HG basis functions on grid of 2048 points on device cpu


Define useful functions

In [44]:
def time_to_trunc_hg(A, hg_basis, dt, num_modes):
    # takes as input a batch of pulses (B, Nt) and returns the HG coefficients (B, B) in the truncated basis
    if A.ndim == 1: # deal with unbatched input
        return time_to_hg(A, hg_basis[:num_modes, :], dt)
    else:
        B = A.shape[0]
        return torch.stack([time_to_hg(A[i], hg_basis[:num_modes, :], dt) for i in range(B)])  # shape: (B, B)

## Define Training Batches

How to form an arbitrary unitary $U$:


1. Start with a Hermitian $H$ for which $H = H^{\dagger}$ (`256` _real_ degrees of freedom)
    - Diagonal entries must be real $h_{ii}\in \R$ (can choose `16` _real_ diagonal entries)
    - Off diagonal entries must satisfy $h_{ij} = \bar{h_{ji}}$ (can choose `120` independent _complex_ entries)
2. exponentiate the Hermitian to obtain the unitary $U = e^{iH}$

In [45]:
# set same random seed to generate random results in a consistent manner
torch.manual_seed(27)

# 1 identity
identity = torch.eye(num_modes, dtype=torch.float32, device=device)

# 2 permutation
# Generate a random permutation of the columns of an identity matrix
idx = torch.randperm(num_modes, device=device)
permutation = torch.eye(num_modes, dtype=torch.float32, device=device)[:, idx]

# 3 rotation
theta = torch.rand(1, device=device) * 2 * torch.pi  # random angle in [0, 2pi)
cos_theta = torch.cos(theta)
sin_theta = torch.sin(theta)
rotation_2x2 = torch.zeros((2, 2), dtype=torch.float32, device=device)
rotation_2x2[0, 0] = cos_theta
rotation_2x2[0, 1] = -sin_theta
rotation_2x2[1, 0] = sin_theta
rotation_2x2[1, 1] = cos_theta
rotation = torch.zeros((16, 16), dtype=torch.float32, device=device) # Create a 16x16 block-diagonal matrix with the 2x2 rotation repeated along the diagonal
for block_idx in range(8):
    start = block_idx * 2
    rotation[start:start+2, start:start+2] = rotation_2x2

# 4 arbitrary
H = torch.zeros((num_modes, num_modes), dtype=torch.cfloat, device=device)
diag_real = torch.randn(num_modes, device=device).to(torch.cfloat)
H[torch.arange(num_modes), torch.arange(num_modes)] = diag_real
for i in range(num_modes):
    for j in range(i+1, num_modes):
        re = torch.randn(1, device=device)
        im = torch.randn(1, device=device)
        val = re + 1j * im
        H[i, j] = val
        H[j, i] = val.conj()
arbitrary = torch.matrix_exp(1j * H)


def is_unitary(matrix, atol=1e-5): # within 1e-5
    identity = torch.eye(16, dtype=matrix.dtype, device=matrix.device)
    if torch.is_complex(matrix):
        prod = matrix @ matrix.conj().T
    else:
        prod = matrix @ matrix.T
    return torch.allclose(prod, identity, atol=atol)

transformations = {
    "identity": identity,
    "permutation": permutation,
    "rotation": rotation,
    "arbitrary": arbitrary
}

# test if all transformations are unitary
for name, transformation in transformations.items():
    print(f"{name}: {is_unitary(transformation)}")
    
# set chosen transformation
U = transformations[t_name].to(torch.cfloat)

identity: True
permutation: True
rotation: True
arbitrary: True


Define basis training batch

In [ ]:
x_basis_train = hg_basis[:num_modes, :] # input basis batch (num_modes, Nt)
x_basis_train_hg = torch.eye(num_modes, dtype=torch.cfloat)
y_basis_train_hg = U @ x_basis_train_hg  # shape: (num_modes, Nt)
y_basis_train = torch.stack([hg_to_time(y_basis_train_hg[i], hg_basis[:num_modes, :]) for i in range(num_modes)]) 